# DocuRAG — Core RAG Pipeline Experiments

This notebook is an **experimentation notebook**, not the production application.

Its purpose is to build and understand every stage of the DocuRAG retrieval pipeline
**before** we wrap it into `src/` modules and a Streamlit app:

`PDF → text extraction → cleaning → chunking → embeddings → FAISS index → retrieval → RAG prompt → LLM answer → source attribution`

There is **no model training** anywhere in this notebook. We only:
- use PyMuPDF to read a PDF,
- use a pretrained local embedding model (`sentence-transformers/all-MiniLM-L6-v2`) for inference,
- use FAISS as a local vector index (not a trained model — an exact search data structure),
- optionally call the Gemini API for generation.

**Note on the sample PDF:** to keep this notebook fully self-contained and runnable without
needing you to find and download a sample paper, we *generate* a short synthetic
"research paper" PDF using PyMuPDF itself. It has a title, authors, an abstract, and four
sections spread across 4 pages — enough structure to meaningfully demonstrate page-aware
chunking, retrieval, and source attribution. When the real application is built, the user
will upload their own PDF instead.


## 1. Environment Setup

Run this once. If you already have these packages installed, `pip` will simply confirm that.

Packages used in this notebook and why:

| Package | Purpose |
|---|---|
| `pymupdf` | Read PDFs, extract text page by page |
| `sentence-transformers` | Load the local embedding model |
| `faiss-cpu` | Local vector index for similarity search |
| `numpy` / `pandas` | Numerical work and tabular inspection |
| `python-dotenv` | Load `GEMINI_API_KEY` from a `.env` file |
| `google-genai` | Official Gemini API client (current SDK) |


In [1]:
%pip install -q --break-system-packages pymupdf sentence-transformers faiss-cpu numpy pandas python-dotenv google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 46.7 MB/s eta 0:00:00


## 2. Imports

In [2]:
import os
import re
import json
from pathlib import Path

import numpy as np
import pandas as pd
import faiss
import pymupdf  # PyMuPDF. Note: the older `import fitz` alias is deprecated.
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

# Current Gemini SDK is `google-genai` (the older `google-generativeai` package
# is deprecated). We only import it here; whether we actually call the API
# depends on a GEMINI_API_KEY being available (see Section 9).
from google import genai
from google.genai import types as genai_types

pd.set_option("display.max_colwidth", 100)
print("Imports OK")

Imports OK


## 3. Create / Load a Sample PDF

We generate a small synthetic "paper" PDF with PyMuPDF so the notebook works offline and
reproducibly. It intentionally includes a page header and footer (e.g. "Page 2 of 4") so we
can demonstrate text-cleaning later — this mimics a very common issue with real PDFs.

If you'd rather experiment with your own PDF, just change `PDF_PATH` below to point at it and
skip running the generation cell.


In [3]:
DATA_DIR = Path("../data/documents")
DATA_DIR.mkdir(parents=True, exist_ok=True)
PDF_PATH = DATA_DIR / "sample_paper.pdf"

PAGES_CONTENT = [
    # Page 1 — Title / authors / abstract
    '''DocuRAG: A Hybrid Retrieval-Augmented System for PDF Document Intelligence

Authors: A. Sharma, R. Menon, K. Fernandes
Department of Computer Science, Sample University

Abstract
This paper presents DocuRAG, a hybrid retrieval-augmented generation system designed to
answer questions over uploaded PDF documents while remaining grounded in the source text.
The system combines local dense retrieval over document chunks with an optional web
retrieval component for queries that require external or up-to-date information. We describe
the chunking strategy, the embedding model used, the vector indexing approach, and a
lightweight query routing mechanism that decides whether a question should be answered
from the document, the web, or both.''',
    # Page 2 — Introduction / problem statement
    '''1. Introduction

Large language models are capable of producing fluent answers, but they frequently
fabricate facts when the required information is not present in their training data or in the
provided context. This problem, commonly called hallucination, is especially problematic
in document question-answering settings, where users expect answers to be traceable back
to a specific page or passage of the source document.

1.1 Problem Statement

The core problem addressed in this work is: how can a system answer natural language
questions about a specific PDF document while (a) remaining grounded in the document's
actual content, (b) citing the page number where the supporting evidence was found, and
(c) transparently falling back to external web search only when the document does not
contain the required information.''',
    # Page 3 — Methodology / dataset
    '''2. Methodology

DocuRAG follows a standard retrieval-augmented generation pipeline. The PDF is first
parsed page by page using PyMuPDF, preserving page numbers as metadata. Extracted text
is cleaned to remove repeated headers, footers, and irrelevant whitespace. The cleaned text
is then split into overlapping chunks. Each chunk is converted into a 384-dimensional dense
vector using the sentence-transformers/all-MiniLM-L6-v2 embedding model, which runs
locally on CPU. The resulting vectors are stored in a FAISS flat index for exact nearest
neighbor search using cosine similarity.

2.1 Dataset

For evaluation of the retrieval component, we constructed a small internal evaluation set of
question-page pairs derived from this sample PDF document. No external benchmark dataset
was used, since the goal of this project is to demonstrate the retrieval and generation
pipeline rather than to outperform existing academic RAG benchmarks such as Natural
Questions or HotpotQA.''',
    # Page 4 — Results / limitations / conclusion
    '''3. Results and Discussion

Using a chunk size of approximately 90 words with 15% overlap and top-k set to 3, the
retrieval component achieved a Hit@3 score on our small internal evaluation set that we
compute later in this notebook, meaning that for a fraction of questions at least one of the
top three retrieved chunks contained the page with the supporting evidence.

3.1 Limitations

The evaluation set is small and manually constructed, so the reported Hit@K numbers should
not be interpreted as a rigorous benchmark result. The current chunking strategy operates
independently on each page and does not merge sentences that are split across a page
boundary, which can occasionally reduce retrieval quality for facts that appear at the very
end or beginning of a page. Scanned, image-based PDFs are not supported in this version,
since no OCR step is implemented.

4. Conclusion

This paper described the design of DocuRAG, a locally runnable hybrid RAG system for PDF
question answering with page-level source attribution. Future work includes supporting
multiple documents at once and adding a proper OCR fallback for scanned documents.

Keywords: retrieval-augmented generation, semantic search, FAISS, sentence embeddings,
hallucination control, document question answering''',
]

def build_sample_pdf(path: Path, pages_content: list[str]) -> None:
    doc = pymupdf.open()
    text_rect = pymupdf.Rect(56, 70, 540, 780)
    for i, page_text in enumerate(pages_content):
        page = doc.new_page(width=595, height=842)  # A4
        page.insert_text((56, 40), "DocuRAG Sample Paper", fontsize=9, color=(0.5, 0.5, 0.5))
        page.insert_textbox(text_rect, page_text, fontsize=11, fontname="helv")
        page.insert_text((260, 810), f"Page {i + 1} of {len(pages_content)}", fontsize=8, color=(0.5, 0.5, 0.5))
    doc.save(str(path))
    doc.close()

if not PDF_PATH.exists():
    build_sample_pdf(PDF_PATH, PAGES_CONTENT)
    print(f"Generated sample PDF at: {PDF_PATH}")
else:
    print(f"Using existing PDF at: {PDF_PATH}")

Generated sample PDF at: ../data/documents/sample_paper.pdf


## 4. Extract Text Page by Page (PyMuPDF)

**Why page-by-page and not the whole document at once?** We need to remember *which page*
each piece of text came from so that later, when the system answers a question, it can point
back to "Page 4" instead of just handing over an unverifiable answer. Page number is the
foundation of source attribution in this project.


In [4]:
def extract_pages(pdf_path: Path) -> list[dict]:
    doc = pymupdf.open(str(pdf_path))
    pages = []
    for i, page in enumerate(doc):
        text = page.get_text("text")
        pages.append({"page": i + 1, "text": text})
    doc.close()
    return pages

raw_pages = extract_pages(PDF_PATH)

if all(len(p["text"].strip()) == 0 for p in raw_pages):
    raise ValueError(
        "No extractable text was found in this PDF. It is likely a scanned / "
        "image-based document, which would require OCR (not implemented in v1)."
    )

print(f"Extracted {len(raw_pages)} pages")
for p in raw_pages:
    print(f"  Page {p['page']}: {len(p['text'])} characters")

Extracted 4 pages
  Page 1: 774 characters
  Page 2: 860 characters
  Page 3: 1007 characters
  Page 4: 1310 characters


## 5. Inspect Extracted Text

A quick sanity check before we do anything else with the text — this is where you'd notice
obvious extraction problems (garbled text, empty pages, repeated headers/footers, etc.).


In [5]:
print("--- Raw text of Page 1 (first 400 chars) ---")
print(raw_pages[0]["text"][:400])
print("\n--- Raw text of Page 4 (first 400 chars) ---")
print(raw_pages[3]["text"][:400])

--- Raw text of Page 1 (first 400 chars) ---
DocuRAG Sample Paper
DocuRAG: A Hybrid Retrieval-Augmented System for PDF Document Intelligence
Authors: A. Sharma, R. Menon, K. Fernandes
Department of Computer Science, Sample University
Abstract
This paper presents DocuRAG, a hybrid retrieval-augmented generation system designed to
answer questions over uploaded PDF documents while remaining grounded in the source text.
The system combines loca

--- Raw text of Page 4 (first 400 chars) ---
DocuRAG Sample Paper
3. Results and Discussion
Using a chunk size of approximately 90 words with 15% overlap and top-k set to 3, the
retrieval component achieved a Hit@3 score on our small internal evaluation set that we
compute later in this notebook, meaning that for a fraction of questions at least one of the
top three retrieved chunks contained the page with the supporting evidence.
3.1 Limita


## 6. Basic Text Cleaning

Real PDFs commonly contain repeated headers/footers, page numbers, and irregular whitespace
from how the PDF renderer laid out the text. None of this is useful content, and if left in,
it wastes tokens and can dilute the embedding signal for a chunk. We apply light,
conservative cleaning:

- remove the repeated page header/footer patterns
- collapse repeated whitespace / blank lines
- strip leading/trailing whitespace

We deliberately do **not** try to do anything aggressive (like removing all newlines or
"fixing" hyphenation), since over-cleaning can also destroy meaning. In v1 we only handle the
patterns we can reliably recognize.


In [6]:
def clean_text(text: str) -> str:
    # Our synthetic PDF always repeats this header on every page — a real PDF's
    # header/footer pattern would need to be identified per-document.
    text = re.sub(r"DocuRAG Sample Paper", "", text)
    text = re.sub(r"Page \d+ of \d+", "", text)

    text = re.sub(r"[ \t]+", " ", text)      # collapse repeated spaces/tabs
    text = re.sub(r"\n{2,}", "\n", text)     # collapse repeated blank lines
    text = text.strip()
    return text

cleaned_pages = [{"page": p["page"], "text": clean_text(p["text"])} for p in raw_pages]

print("--- Cleaned text of Page 1 ---")
print(cleaned_pages[0]["text"][:400])

--- Cleaned text of Page 1 ---
DocuRAG: A Hybrid Retrieval-Augmented System for PDF Document Intelligence
Authors: A. Sharma, R. Menon, K. Fernandes
Department of Computer Science, Sample University
Abstract
This paper presents DocuRAG, a hybrid retrieval-augmented generation system designed to
answer questions over uploaded PDF documents while remaining grounded in the source text.
The system combines local dense retrieval ove


## 7. Chunking

**Why do we chunk at all, instead of embedding whole pages or the whole document?**
Embedding models produce a *single* fixed-size vector per input. If we embed an entire page
(or the whole PDF), that one vector has to represent many different ideas at once, so
similarity search becomes blunt — a highly relevant sentence gets diluted by everything else
on the page. Smaller, focused chunks give more precise retrieval.

**Why overlap?** If we chunk with hard, non-overlapping boundaries, a fact that happens to
sit right at a chunk boundary can get split across two chunks and become hard to retrieve
in full from either one. A small overlap (here 10–20%) lets neighboring chunks share some
context so boundary information isn't lost.

**Baseline used in this project:** ~500–800 words per chunk with ~10–20% overlap is a
reasonable starting point for real-world documents (this is *not* a universally "correct"
number — it depends on document type and embedding model). Because our sample PDF is very
short (4 pages, a few hundred words each), we use a **smaller chunk size (90 words)** here
purely so that a short demo document still produces multiple chunks per page to search over.
When we point this pipeline at real, longer documents later, `CHUNK_SIZE` should move back
toward the 500–800 word range — Section 15 lets you experiment with this directly.

Every chunk keeps its page number and a unique ID — this is what makes source attribution
possible later.


In [7]:
CHUNK_SIZE = 90          # words per chunk (demo value — see note above)
CHUNK_OVERLAP_RATIO = 0.15  # 15% overlap between consecutive chunks
DOCUMENT_ID = "sample_paper"

def chunk_page_text(text: str, page: int, chunk_size: int, overlap_ratio: float, document_id: str) -> list[dict]:
    words = text.split()
    if not words:
        return []

    step = max(1, int(chunk_size * (1 - overlap_ratio)))
    chunks = []
    start = 0
    chunk_num = 0
    while start < len(words):
        window = words[start:start + chunk_size]
        if not window:
            break
        chunks.append({
            "text": " ".join(window),
            "page": page,
            "chunk_id": f"{document_id}_p{page}_c{chunk_num}",
            "document_id": document_id,
            "word_count": len(window),
        })
        chunk_num += 1
        if start + chunk_size >= len(words):
            break
        start += step
    return chunks

def chunk_document(pages: list[dict], chunk_size: int, overlap_ratio: float, document_id: str) -> list[dict]:
    # We chunk each page independently. This keeps page-attribution simple and correct,
    # at the cost of occasionally splitting a sentence that straddles a page break —
    # a known limitation discussed in Section 21.
    all_chunks = []
    for p in pages:
        all_chunks.extend(chunk_page_text(p["text"], p["page"], chunk_size, overlap_ratio, document_id))
    return all_chunks

all_chunks = chunk_document(cleaned_pages, CHUNK_SIZE, CHUNK_OVERLAP_RATIO, DOCUMENT_ID)
print(f"Produced {len(all_chunks)} chunks from {len(cleaned_pages)} pages")

Produced 9 chunks from 4 pages


## 8. Inspect Chunks and Metadata

Every chunk carries exactly the metadata described in the project spec: `text`, `page`,
`chunk_id`, `document_id`. This is the structure that will later be stored alongside the
FAISS index so we can map a retrieved vector ID back to real content.


In [8]:
chunks_df = pd.DataFrame(all_chunks)[["chunk_id", "page", "word_count", "text"]].copy()
chunks_df["text_preview"] = chunks_df["text"].str.slice(0, 70) + "..."
print(chunks_df[["chunk_id", "page", "word_count", "text_preview"]].to_string(index=False))

          chunk_id  page  word_count                                                              text_preview
sample_paper_p1_c0     1          90 DocuRAG: A Hybrid Retrieval-Augmented System for PDF Document Intellig...
sample_paper_p1_c1     1          28 embedding model used, the vector indexing approach, and a lightweight ...
sample_paper_p2_c0     2          90 1. Introduction Large language models are capable of producing fluent ...
sample_paper_p2_c1     2          49 system answer natural language questions about a specific PDF document...
sample_paper_p3_c0     3          90 2. Methodology DocuRAG follows a standard retrieval-augmented generati...
sample_paper_p3_c1     3          64 nearest neighbor search using cosine similarity. 2.1 Dataset For evalu...
sample_paper_p4_c0     4          90 3. Results and Discussion Using a chunk size of approximately 90 words...
sample_paper_p4_c1     4          90 Hit@K numbers should not be interpreted as a rigorous benchmark result...
s

In [9]:
print("Example chunk (page 3):")
example = next(c for c in all_chunks if c["page"] == 3)
print(json.dumps(example, indent=2))

Example chunk (page 3):
{
  "text": "2. Methodology DocuRAG follows a standard retrieval-augmented generation pipeline. The PDF is first parsed page by page using PyMuPDF, preserving page numbers as metadata. Extracted text is cleaned to remove repeated headers, footers, and irrelevant whitespace. The cleaned text is then split into overlapping chunks. Each chunk is converted into a 384-dimensional dense vector using the sentence-transformers/all-MiniLM-L6-v2 embedding model, which runs locally on CPU. The resulting vectors are stored in a FAISS flat index for exact nearest neighbor search using cosine similarity. 2.1 Dataset For evaluation of the retrieval component,",
  "page": 3,
  "chunk_id": "sample_paper_p3_c0",
  "document_id": "sample_paper",
  "word_count": 90
}


## 9. Load the Embedding Model Locally

We use `sentence-transformers/all-MiniLM-L6-v2`:
- small (~80MB), fast on CPU — appropriate for a free, locally-runnable project
- maps any input text to a **384-dimensional** dense vector
- trained so that semantically similar sentences end up close together in vector space,
  even if they don't share the same words

**Intuition:** think of the embedding as coordinates in a 384-dimensional "meaning space".
"What dataset was used?" and "which data source was used for evaluation?" use almost no
words in common, but a good embedding model places them near each other because they mean
similar things.

The first time this runs it downloads the model from Hugging Face and caches it locally
(usually under `~/.cache/huggingface`); every run after that loads instantly from disk —
no re-download, no training involved.


In [10]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(f"Loaded embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Max sequence length: {embedding_model.max_seq_length} tokens")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded embedding model: sentence-transformers/all-MiniLM-L6-v2
Max sequence length: 256 tokens


## 10. Generate Embeddings for Chunks

In [11]:
chunk_texts = [c["text"] for c in all_chunks]
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    show_progress_bar=False,
).astype("float32")

print(f"chunk_embeddings shape: {chunk_embeddings.shape}")

chunk_embeddings shape: (9, 384)


## 11. Inspect Embedding Dimensions

`chunk_embeddings.shape` should be `(number_of_chunks, 384)` — one 384-dimensional vector
per chunk. This dimensionality is fixed by the model architecture, not something we choose.


In [12]:
num_chunks, embedding_dim = chunk_embeddings.shape
print(f"{num_chunks} chunks, each represented as a {embedding_dim}-dimensional vector")
print("First 8 values of the first chunk's vector:")
print(chunk_embeddings[0][:8])

9 chunks, each represented as a 384-dimensional vector
First 8 values of the first chunk's vector:
[-0.07347771  0.00375272 -0.06520756  0.11445296  0.0365511   0.03027412
 -0.03613885  0.03608402]


## 12. Build a FAISS Index

FAISS is **not** a trained model — it's a local library for fast (here, exact) nearest-
neighbor search over vectors. We use `IndexFlatIP` (Inner Product) rather than the more
common `IndexFlatL2` (Euclidean distance), because:

- if we **L2-normalize** every vector (unit length) first,
- then the **inner product** between two normalized vectors is mathematically identical to
  their **cosine similarity**.

Cosine similarity measures the *angle* between two vectors rather than their raw distance,
which works well for sentence embeddings, where the direction of the vector carries the
semantic meaning more reliably than its magnitude.

We also keep a simple Python list, `chunk_metadata`, where position `i` corresponds exactly
to FAISS vector ID `i`. This is the mapping the project spec requires: **FAISS vector ID →
chunk ID → page number → text**. Without this mapping, a similarity search would return
"vector #7 is the best match" and nothing else — useless without a way to look up what that
vector represents.


In [13]:
def build_faiss_index(embeddings: np.ndarray) -> faiss.IndexFlatIP:
    normalized = embeddings.copy().astype("float32")
    faiss.normalize_L2(normalized)  # in-place L2 normalization
    index = faiss.IndexFlatIP(normalized.shape[1])
    index.add(normalized)
    return index

faiss_index = build_faiss_index(chunk_embeddings)
chunk_metadata = all_chunks  # chunk_metadata[i] describes FAISS vector ID i

print(f"FAISS index built. Total vectors indexed: {faiss_index.ntotal}")

FAISS index built. Total vectors indexed: 9


## 13. Encode a Sample Question and Search

To search, we embed the user's question with the **same** embedding model used for the
chunks (this is essential — query and document vectors must live in the same space), then
ask FAISS for the `TOP_K` chunks whose vectors are most similar to the question's vector.


In [14]:
TOP_K = 3

def embed_query(query: str, model: SentenceTransformer) -> np.ndarray:
    q = model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q)
    return q

def search(query: str, model: SentenceTransformer, index: faiss.IndexFlatIP,
           metadata: list[dict], top_k: int) -> list[dict]:
    q_vec = embed_query(query, model)
    scores, ids = index.search(q_vec, top_k)
    results = []
    for score, idx in zip(scores[0], ids[0]):
        if idx == -1:
            continue
        chunk = metadata[idx]
        results.append({**chunk, "similarity_score": float(score)})
    return results

sample_question = "What dataset was used to evaluate the retrieval component?"
results = search(sample_question, embedding_model, faiss_index, chunk_metadata, TOP_K)
print(f"Question: {sample_question}\n")

Question: What dataset was used to evaluate the retrieval component?



## 14. Display Retrieved Chunks, Page Numbers, and Similarity Scores

In [15]:
results_df = pd.DataFrame(results)[["chunk_id", "page", "similarity_score", "text"]].copy()
results_df["similarity_score"] = results_df["similarity_score"].round(4)
results_df["text_preview"] = results_df["text"].str.slice(0, 80) + "..."
print(results_df[["chunk_id", "page", "similarity_score", "text_preview"]].to_string(index=False))

          chunk_id  page  similarity_score                                                                        text_preview
sample_paper_p4_c0     4            0.5563 3. Results and Discussion Using a chunk size of approximately 90 words with 15% ...
sample_paper_p3_c0     3            0.5205 2. Methodology DocuRAG follows a standard retrieval-augmented generation pipelin...
sample_paper_p3_c1     3            0.4598 nearest neighbor search using cosine similarity. 2.1 Dataset For evaluation of t...


## 15. Experiment: Chunk Size, Overlap, and Top-K

These three parameters directly control the recall/precision trade-off of retrieval:

- **Chunk size ↓** → more precise, focused chunks, but each chunk has less surrounding
  context, and long documents produce many more chunks (more storage, slightly slower search).
- **Chunk size ↑** → more context per chunk, but a chunk can drift across multiple topics,
  which makes its single embedding a blurrier average of everything in it.
- **Overlap ↑** → less chance of losing a fact that sits at a chunk boundary, at the cost of
  redundant storage (the same words get embedded more than once).
- **Top-K ↑** → higher chance the right chunk is *somewhere* in the results (**recall**), but
  more irrelevant text gets sent to the LLM as context, which can dilute or confuse the
  answer (**precision** drops).

Below we rebuild the whole pipeline (chunk → embed → index) under a couple of different
configurations and compare what gets retrieved for the same question, so you can see these
trade-offs directly rather than just being told about them.


In [16]:
def run_pipeline(pages: list[dict], chunk_size: int, overlap_ratio: float,
                  model: SentenceTransformer, document_id: str = "sample_paper"):
    chunks = chunk_document(pages, chunk_size, overlap_ratio, document_id)
    embeddings = model.encode([c["text"] for c in chunks], convert_to_numpy=True).astype("float32")
    index = build_faiss_index(embeddings)
    return chunks, index

experiment_configs = [
    {"chunk_size": 40, "overlap_ratio": 0.15, "top_k": 3, "label": "small chunks (40 words)"},
    {"chunk_size": 90, "overlap_ratio": 0.15, "top_k": 3, "label": "baseline (90 words)"},
    {"chunk_size": 90, "overlap_ratio": 0.15, "top_k": 6, "label": "baseline, higher top_k=6"},
]

question = "What is the Hit@3 score reported for retrieval?"

for cfg in experiment_configs:
    exp_chunks, exp_index = run_pipeline(cleaned_pages, cfg["chunk_size"], cfg["overlap_ratio"], embedding_model)
    exp_results = search(question, embedding_model, exp_index, exp_chunks, cfg["top_k"])
    print(f"--- {cfg['label']} | total_chunks={len(exp_chunks)} ---")
    for r in exp_results:
        snippet = r["text"][:90].replace("\n", " ")
        print(f"  page={r['page']}  score={r['similarity_score']:.4f}  \"{snippet}...\"")
    print()

--- small chunks (40 words) | total_chunks=17 ---
  page=4  score=0.5974  "3. Results and Discussion Using a chunk size of approximately 90 words with 15% overlap an..."
  page=4  score=0.4880  "is small and manually constructed, so the reported Hit@K numbers should not be interpreted..."
  page=3  score=0.4190  "this sample PDF document. No external benchmark dataset was used, since the goal of this p..."

--- baseline (90 words) | total_chunks=9 ---
  page=4  score=0.6318  "3. Results and Discussion Using a chunk size of approximately 90 words with 15% overlap an..."
  page=4  score=0.3581  "Hit@K numbers should not be interpreted as a rigorous benchmark result. The current chunki..."
  page=3  score=0.3467  "nearest neighbor search using cosine similarity. 2.1 Dataset For evaluation of the retriev..."

--- baseline, higher top_k=6 | total_chunks=9 ---
  page=4  score=0.6318  "3. Results and Discussion Using a chunk size of approximately 90 words with 15% overlap an..."
  page=4  sco

## 16. From Retrieved Chunks to RAG Context

Before we can ask an LLM to answer, we assemble the retrieved chunks into a single block of
**context text**, tagging each passage with its page number. This inline page-tagging is what
lets both the LLM (for citing pages in its answer) and our own code (for the final
"Sources:" list) know exactly which page each piece of evidence came from.


In [17]:
def build_context(results: list[dict]) -> str:
    parts = []
    for r in results:
        parts.append(f"[Page {r['page']}]\n{r['text']}")
    return "\n\n".join(parts)

context_text = build_context(results)
print(context_text)

[Page 4]
3. Results and Discussion Using a chunk size of approximately 90 words with 15% overlap and top-k set to 3, the retrieval component achieved a Hit@3 score on our small internal evaluation set that we compute later in this notebook, meaning that for a fraction of questions at least one of the top three retrieved chunks contained the page with the supporting evidence. 3.1 Limitations The evaluation set is small and manually constructed, so the reported Hit@K numbers should not be interpreted as a rigorous benchmark result. The current chunking

[Page 3]
2. Methodology DocuRAG follows a standard retrieval-augmented generation pipeline. The PDF is first parsed page by page using PyMuPDF, preserving page numbers as metadata. Extracted text is cleaned to remove repeated headers, footers, and irrelevant whitespace. The cleaned text is then split into overlapping chunks. Each chunk is converted into a 384-dimensional dense vector using the sentence-transformers/all-MiniLM-L6-v2 embedd

## 17. RAG Prompt: What Gets Sent to Gemini

We separate a **system prompt** (grounding rules, always the same) from a **user prompt**
(the retrieved context + the actual question, different every time). The grounding rules are
the main defense against hallucination for PDF-only questions: the model is explicitly told
to answer only from the given context and to say so plainly when the context doesn't cover
the question, rather than guessing.


In [18]:
SYSTEM_PROMPT = '''You are DocuRAG, a document question-answering assistant.
Answer the user's question using ONLY the information in the PDF CONTEXT below.
Each context passage is labeled with the page number it came from.

Rules:
- If the answer is not supported by the given context, say exactly:
  "I could not find this information in the uploaded document."
- Do not use any outside knowledge.
- Do not invent page numbers, authors, numbers, or facts that are not in the context.
- When you state a fact, mention which page it came from, e.g. (Page 3).
'''

USER_PROMPT_TEMPLATE = '''PDF CONTEXT:
{context}

QUESTION:
{question}

Answer using only the PDF CONTEXT above, and cite page numbers for every claim.
'''

def build_user_prompt(context: str, question: str) -> str:
    return USER_PROMPT_TEMPLATE.format(context=context, question=question)

example_prompt = build_user_prompt(context_text, sample_question)
print("=== SYSTEM PROMPT ===")
print(SYSTEM_PROMPT)
print("=== USER PROMPT (what would be sent alongside the system prompt) ===")
print(example_prompt)

=== SYSTEM PROMPT ===
You are DocuRAG, a document question-answering assistant.
Answer the user's question using ONLY the information in the PDF CONTEXT below.
Each context passage is labeled with the page number it came from.

Rules:
- If the answer is not supported by the given context, say exactly:
  "I could not find this information in the uploaded document."
- Do not use any outside knowledge.
- Do not invent page numbers, authors, numbers, or facts that are not in the context.
- When you state a fact, mention which page it came from, e.g. (Page 3).

=== USER PROMPT (what would be sent alongside the system prompt) ===
PDF CONTEXT:
[Page 4]
3. Results and Discussion Using a chunk size of approximately 90 words with 15% overlap and top-k set to 3, the retrieval component achieved a Hit@3 score on our small internal evaluation set that we compute later in this notebook, meaning that for a fraction of questions at least one of the top three retrieved chunks contained the page with th

## 18. Generate a Grounded Answer with Gemini

We load `GEMINI_API_KEY` from a `.env` file (never hardcoded — see `.env.example` created in
a later phase). This notebook is meant to also work for readers who haven't set up a key yet,
so if no key is found we skip the live call rather than pretending to have an answer — the
prompt structure above is exactly what `llm.py` will send once wired into the application in
Phase 8.

We use `gemini-2.5-flash` as a fast, low-cost model suitable for this kind of grounded
Q&A task. **Verify current Gemini API pricing/free-tier limits at
[ai.google.dev](https://ai.google.dev) before relying on this in production** — free tier
availability and quotas change over time and should not be assumed permanent.


In [19]:
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_MODEL = "gemini-2.5-flash"

def ask_gemini(system_prompt: str, user_prompt: str, api_key: str, model: str = GEMINI_MODEL) -> str | None:
    try:
        client = genai.Client(api_key=api_key)
        response = client.models.generate_content(
            model=model,
            contents=user_prompt,
            config=genai_types.GenerateContentConfig(
                system_instruction=system_prompt,
                temperature=0.2,
            ),
        )
        return response.text
    except Exception as e:
        print(f"Gemini API call failed: {type(e).__name__}: {e}")
        return None

if GEMINI_API_KEY:
    answer = ask_gemini(SYSTEM_PROMPT, example_prompt, GEMINI_API_KEY)
    if answer:
        print("=== Grounded Answer ===")
        print(answer)
else:
    answer = None
    print(
        "No GEMINI_API_KEY found in the environment.\n"
        "This cell shows the pipeline structure only — it will produce a real grounded "
        "answer once a key is added to a .env file (Phase 8 of the project).\n"
        "Nothing is fabricated here in place of a real model response."
    )

No GEMINI_API_KEY found in the environment.
This cell shows the pipeline structure only — it will produce a real grounded answer once a key is added to a .env file (Phase 8 of the project).
Nothing is fabricated here in place of a real model response.


## 19. Source / Page Attribution

Source attribution must always be derived from the chunks that were **actually retrieved**,
never guessed or reconstructed from the answer text. Since every chunk already carries its
page number, this is just a matter of collecting and de-duplicating pages from `results`.


In [20]:
def format_sources(results: list[dict]) -> str:
    pages = sorted({r["page"] for r in results})
    return "\n".join(f"- PDF — Page {p}" for p in pages)

print("Sources:")
print(format_sources(results))

Sources:
- PDF — Page 3
- PDF — Page 4


## 20. Document Summarization Experiment (Map-Reduce Style)

For long documents we must **not** stuff the entire PDF into a single LLM call — it may
exceed context limits, cost more, and tends to produce vaguer summaries. Instead we use a
hierarchical / map-reduce strategy:

1. **Map:** summarize each chunk (or page) independently → *intermediate summaries*.
2. **Reduce:** combine the intermediate summaries into one *final structured summary*.

Our sample document is short enough to summarize directly, but the pattern below scales to
much longer documents by simply increasing the number of intermediate summaries combined in
the reduce step. This is the same pattern `summarizer.py` will implement in Phase 10.


In [21]:
CHUNK_SUMMARY_PROMPT = '''Summarize the following excerpt from a document in 1-2 sentences.
Only include information that is explicitly present in the excerpt. Do not add outside knowledge.

Excerpt:
{chunk_text}
'''

FINAL_SUMMARY_PROMPT = '''You are given several short summaries of different parts of the same document.
Combine them into a single structured summary with these fields:
- Title
- Authors
- Objective
- Methodology
- Key Findings
- Limitations
- Conclusion

If a field cannot be determined from the summaries below, write exactly:
"Not clearly identified in the document."

Partial summaries:
{combined_summaries}
'''

def summarize_document_map_reduce(chunks: list[dict], api_key: str | None) -> str | None:
    if not api_key:
        print(
            "No GEMINI_API_KEY found — skipping live summarization.\n"
            "Below are the exact prompts that would be sent for the map step (per chunk) "
            "and the reduce step (combining all chunk summaries)."
        )
        print("\n--- Example MAP prompt (chunk 0) ---")
        print(CHUNK_SUMMARY_PROMPT.format(chunk_text=chunks[0]["text"]))
        return None

    # Map step: summarize each chunk
    intermediate_summaries = []
    for c in chunks:
        prompt = CHUNK_SUMMARY_PROMPT.format(chunk_text=c["text"])
        summary = ask_gemini("You are a precise, factual summarizer.", prompt, api_key)
        if summary:
            intermediate_summaries.append(f"(Page {c['page']}) {summary.strip()}")

    # Reduce step: combine into one structured summary
    combined = "\n".join(intermediate_summaries)
    final_prompt = FINAL_SUMMARY_PROMPT.format(combined_summaries=combined)
    return ask_gemini("You are a precise, factual summarizer.", final_prompt, api_key)

final_summary = summarize_document_map_reduce(all_chunks, GEMINI_API_KEY)
if final_summary:
    print("=== Final Structured Summary ===")
    print(final_summary)

No GEMINI_API_KEY found — skipping live summarization.
Below are the exact prompts that would be sent for the map step (per chunk) and the reduce step (combining all chunk summaries).

--- Example MAP prompt (chunk 0) ---
Summarize the following excerpt from a document in 1-2 sentences.
Only include information that is explicitly present in the excerpt. Do not add outside knowledge.

Excerpt:
DocuRAG: A Hybrid Retrieval-Augmented System for PDF Document Intelligence Authors: A. Sharma, R. Menon, K. Fernandes Department of Computer Science, Sample University Abstract This paper presents DocuRAG, a hybrid retrieval-augmented generation system designed to answer questions over uploaded PDF documents while remaining grounded in the source text. The system combines local dense retrieval over document chunks with an optional web retrieval component for queries that require external or up-to-date information. We describe the chunking strategy, the embedding model used, the vector indexing app

## 21. Basic Retrieval Evaluation (Hit@K)

To know whether retrieval is actually working — not just "the code runs" — we need a small,
honest evaluation set: a handful of questions about our sample document, each paired with the
page(s) we (as the document's authors, in this case) know contain the answer.

**Hit@K** answers a simple question: *for what fraction of test questions does at least one
of the top-K retrieved chunks come from a correct page?* It's a coarse but meaningful signal
of retrieval quality, and it's cheap to compute without needing an LLM at all.


In [22]:
EVAL_SET = [
    {"question": "Who are the authors of this paper?", "expected_pages": [1]},
    {"question": "What embedding model does DocuRAG use locally?", "expected_pages": [3]},
    {"question": "What dataset was used to evaluate retrieval?", "expected_pages": [3]},
    {"question": "What problem does the paper identify with large language models?", "expected_pages": [2]},
    {"question": "What are the limitations discussed in the paper?", "expected_pages": [4]},
    {"question": "What future work is suggested in the conclusion?", "expected_pages": [4]},
]

def hit_at_k(eval_set: list[dict], model: SentenceTransformer, index: faiss.IndexFlatIP,
             metadata: list[dict], k: int) -> tuple[float, pd.DataFrame]:
    rows = []
    hits = 0
    for item in eval_set:
        retrieved = search(item["question"], model, index, metadata, k)
        retrieved_pages = {r["page"] for r in retrieved}
        is_hit = len(retrieved_pages.intersection(item["expected_pages"])) > 0
        hits += int(is_hit)
        rows.append({
            "question": item["question"],
            "expected_pages": item["expected_pages"],
            "retrieved_pages": sorted(retrieved_pages),
            "hit": is_hit,
        })
    score = hits / len(eval_set)
    return score, pd.DataFrame(rows)

hit_score, eval_df = hit_at_k(EVAL_SET, embedding_model, faiss_index, chunk_metadata, TOP_K)
print(eval_df.to_string(index=False))
print(f"\nHit@{TOP_K}: {hit_score:.2f} ({int(hit_score * len(EVAL_SET))}/{len(EVAL_SET)} questions)")

                                                        question expected_pages retrieved_pages   hit
                              Who are the authors of this paper?            [1]          [3, 4] False
                  What embedding model does DocuRAG use locally?            [3]       [1, 3, 4]  True
                    What dataset was used to evaluate retrieval?            [3]          [3, 4]  True
What problem does the paper identify with large language models?            [2]       [2, 3, 4]  True
                What are the limitations discussed in the paper?            [4]          [2, 4]  True
                What future work is suggested in the conclusion?            [4]          [2, 4]  True

Hit@3: 0.83 (5/6 questions)


## 22. Discussion: Limitations of This Notebook's Evaluation

Being explicit about what this evaluation does **not** show is as important as the metric
itself:

- **Tiny, hand-built eval set.** Six questions on a synthetic four-page document is enough to
  sanity-check the pipeline, not to make statistically meaningful claims about retrieval
  quality. A real evaluation set (Phase 17) should have more questions across real documents.
- **Hit@K only measures retrieval, not generation.** A "hit" means a relevant chunk was
  *retrieved* — it says nothing about whether the LLM's final answer was actually correct,
  faithful to that chunk, or well-written. Those would be separate metrics: **context
  relevance** (is the retrieved text actually useful for the question), **answer
  faithfulness** (does the answer only say things supported by the context), and **answer
  correctness** (is the answer actually right) — none of which are measured here.
- **Page-level granularity is coarse.** A hit only confirms the correct *page* was retrieved,
  not necessarily the most useful *chunk* on that page.
- **Chunking does not cross page boundaries**, so a fact split across the end of one page and
  the start of the next may not be fully captured in any single chunk.
- **No OCR** — a scanned PDF with no extractable text would fail at Section 4, by design,
  rather than silently returning nothing.
- Running this same evaluation on a real, longer PDF (a real paper, a report, etc.) is the
  natural next test before trusting these numbers.

## Summary

This notebook demonstrated, end-to-end and without any framework hiding the steps, the full
core RAG pipeline that DocuRAG's application code will implement:

`PDF → page-aware extraction → cleaning → chunking → local embeddings → FAISS index →
similarity search → context construction → grounded LLM prompting → source attribution →
map-reduce summarization → basic retrieval evaluation`

**Next step (Phase 2):** turn Sections 3–4 into `src/pdf_loader.py`.
